# Self-RAG: Reflection-Driven Retrieval and Generation

| Property | Value |
|---|---|
| Origin | Asai et al., *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection* (2024). [arXiv:2310.11511](https://arxiv.org/abs/2310.11511) |

This notebook implements **Self-RAG** (Self-Reflective Retrieval-Augmented Generation), taking inspiration
from the [Self-RAG paper](https://arxiv.org/abs/2310.11511) (Asai et al., 2023) and from the conceptual
implementation in [`FareedKhan-dev/all-agentic-architectures`](https://github.com/FareedKhan-dev/all-agentic-architectures)
(see its `25_self_rag` module — we do not reuse any of that repo's code, only the underlying idea, and we build
our own from-scratch implementation on top of LangGraph and this repo's `helpers` factory).

The central idea of Self-RAG is that a single LLM is repeatedly asked to **critique its own process** at several
distinct decision points, instead of relying on one external router or one external grader:

1. **Should I even retrieve for this query?** — not every question needs external context.
2. **Is each retrieved chunk actually relevant?** — a per-document relevance judgment.
3. **Is my generated answer grounded in the retrieved documents?** — a groundedness / hallucination check.
4. **Does my generated answer actually address the question?** — a utility check.

Each of these is implemented as a small structured-output judgment from the LLM (a *reflection token* in the
original paper's terminology), and each judgment drives a real conditional branch in the graph — so the model is
reflecting on, and correcting, its own retrieval and generation behavior at multiple points in a single pass,
rather than being routed once up front or corrected once via a single fallback.

## How this differs from the other notebooks in this folder

This folder already has two other agentic RAG patterns built with LangGraph. Self-RAG is a genuinely different
design, not a re-skin of either:

| | **2. Corrective RAG (CRAG)** | **3. Adaptive RAG** | **4. Self-RAG (this notebook)** |
|---|---|---|---|
| **Where the "smarts" sit** | One relevance grade over retrieved docs, then a binary fallback | One upfront query router that picks a data source before any retrieval | Several independent self-reflection judgments made by the same LLM at different stages of the pipeline |
| **Decision points** | 1: "are >50% of the docs relevant?" | 1: "which source — vectorstore or web search — fits this query?" | 4+: retrieve-or-not, per-chunk relevance, groundedness-of-generation, utility-of-generation — each with its own conditional edge |
| **Recovery mechanism** | Rewrite the query once and fall back to web search | Route to a different data source; a downstream hallucination check can trigger one regeneration | A bounded self-correction loop: re-retrieve on low relevance, regenerate on poor groundedness, re-retrieve/rephrase on poor utility — each independently, up to an iteration cap |
| **External tools** | Vector DB + web search (Tavily) | Vector DB + web search (Tavily) | Vector DB only — no web search fallback; the "correction" is entirely the model reflecting on and re-doing its own retrieval/generation |
| **Can skip retrieval entirely** | No — always retrieves first | No — always retrieves from *some* source | Yes — the first reflection step can decide retrieval isn't needed at all and answer directly |

In short: CRAG asks "was retrieval good enough, yes/no, then patch it with the web"; Adaptive RAG asks "which
source should I use, before I even retrieve"; Self-RAG asks "should I retrieve at all, is *this* chunk relevant,
is *this* generation supported, is *this* generation useful" — four separate self-critiques, each wired to its
own branch in the graph, with a small iteration cap so the loops always terminate.

## Setup: environment, LLM and embeddings

We use this repo's shared `helpers` factory for both the LLM and the embedding model, so this notebook stays platform-aware (Groq on Windows, Databricks on macOS) and never instantiates a provider client directly.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from helpers import get_llm, get_embeddings

llm = get_llm(temperature=0)
embeddings = get_embeddings()

llm, embeddings

## Build a small, self-contained toy corpus

To keep this notebook runnable end-to-end without external downloads, we index a handful of short, hand-written
passages about agentic AI design patterns directly into an in-memory Chroma vector store. The corpus is
deliberately small and topically narrow — this is what makes the retrieve-decision and relevance-grading steps
downstream meaningful: some queries will have great matches, some will have none at all, and one query won't
need retrieval whatsoever.

In [ ]:
from langchain_core.documents import Document

toy_corpus = [
    Document(
        page_content=(
            "ReAct (Reason + Act) is an agent pattern where a language model interleaves free-form reasoning "
            "traces with tool-calling actions. At each step the model writes a short thought, decides on an "
            "action (such as a search or calculator call), observes the result, and repeats until it can produce "
            "a final answer. This interleaving lets the agent adapt its plan based on real observations instead "
            "of committing to a fixed sequence of steps up front."
        ),
        metadata={"topic": "ReAct"},
    ),
    Document(
        page_content=(
            "Reflexion is an agent pattern that adds a verbal self-critique loop on top of an acting agent. "
            "After attempting a task, the agent (or a separate evaluator) reflects on what went wrong, writes a "
            "short natural-language lesson, and stores it in an episodic memory buffer. On the next attempt the "
            "agent conditions on these past reflections, which lets it improve across multiple tries without any "
            "gradient updates to the underlying model."
        ),
        metadata={"topic": "Reflexion"},
    ),
    Document(
        page_content=(
            "Tree of Thoughts generalizes chain-of-thought prompting by letting a model explore multiple "
            "reasoning branches at once instead of committing to a single linear thought process. The model "
            "generates several candidate next steps, a value function scores each partial solution, and a search "
            "strategy such as breadth-first or depth-first search decides which branches to expand further and "
            "which to prune."
        ),
        metadata={"topic": "Tree of Thoughts"},
    ),
    Document(
        page_content=(
            "Self-RAG is a retrieval-augmented generation approach where a single model is trained (or prompted) "
            "to emit special reflection tokens that control its own pipeline: a retrieve-token decides whether "
            "external context is needed at all, an ISREL-style token grades whether a retrieved passage is "
            "relevant, an ISSUP-style token grades whether a generated segment is supported by the retrieved "
            "passages, and an ISUSE-style token grades whether the overall response is useful for the query. "
            "This lets the same model decide when to retrieve, what to trust, and when to regenerate."
        ),
        metadata={"topic": "Self-RAG"},
    ),
    Document(
        page_content=(
            "Corrective RAG (CRAG) adds a lightweight relevance grader after retrieval from a vector database. "
            "If a large enough fraction of the retrieved documents are graded as relevant, the system generates "
            "an answer directly from them. If not, it rewrites the query to be more web-search friendly and "
            "falls back to a live web search tool, merging any usable web results with whatever relevant "
            "documents were already retrieved before generating the final answer."
        ),
        metadata={"topic": "Corrective RAG"},
    ),
    Document(
        page_content=(
            "Adaptive RAG routes each incoming query, before any retrieval happens, to whichever data source is "
            "best suited to answer it. A classifier prompt looks at the query and decides between a vector "
            "database of indexed documents and a live web search tool. The chosen source is then queried, "
            "documents are graded for relevance, and a hallucination check can trigger a single regeneration "
            "pass if the answer is not grounded in the retrieved context."
        ),
        metadata={"topic": "Adaptive RAG"},
    ),
    Document(
        page_content=(
            "Vector databases store dense embedding representations of documents and support approximate "
            "nearest-neighbor search over them. A user query is embedded with the same embedding model used to "
            "index the documents, and the database returns the stored chunks whose embeddings are closest by a "
            "similarity metric such as cosine similarity. This is the core retrieval mechanism behind almost all "
            "modern RAG pipelines."
        ),
        metadata={"topic": "Vector Databases"},
    ),
    Document(
        page_content=(
            "Prompt engineering covers the practice of designing the instructions, examples, and structure given "
            "to a language model to reliably elicit a desired behavior. Common techniques include few-shot "
            "examples, explicit output-format constraints, chain-of-thought instructions that ask the model to "
            "reason step by step, and system prompts that fix a persona or a set of rules the model should "
            "follow throughout a conversation."
        ),
        metadata={"topic": "Prompt Engineering"},
    ),
]

len(toy_corpus)

In [ ]:
from langchain_chroma import Chroma

# In-memory Chroma collection (no persist_directory) — this is intentionally small and ephemeral,
# rebuilt fresh every time this notebook runs.
vectorstore = Chroma.from_documents(
    documents=toy_corpus,
    collection_name="self_rag_toy_corpus",
    embedding=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
# Quick sanity check on retrieval before we wire up the graph
for doc in retriever.invoke("How does Self-RAG decide when to retrieve?"):
    print(doc.metadata["topic"], "-", doc.page_content[:80], "...")

## Reflection workflow 1 — Retrieve decision

The very first self-reflection point: given only the question (no retrieval has happened yet), should the model
retrieve external context at all? Simple factual/arithmetic/greeting-style queries the model can already answer
confidently don't need it; queries about the toy corpus's specific content do.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class RetrieveDecision(BaseModel):
    """Decision on whether retrieval is needed to answer a question well."""

    need_retrieval: Literal["yes", "no"] = Field(
        description=(
            "'yes' if answering this question well requires looking up external documents, "
            "'no' if the model can already answer it directly and reliably from its own knowledge "
            "(e.g. simple arithmetic, greetings, general definitions it is confident about)."
        )
    )
    reasoning: str = Field(description="One short sentence explaining the decision.")


retrieve_decider = llm.with_structured_output(RetrieveDecision)

RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are the first stage of a Self-RAG pipeline. Decide whether the question below needs "
            "retrieval from an external knowledge base of notes about agentic AI design patterns, or "
            "whether it can be answered directly without any lookup.",
        ),
        ("human", "Question:\n{question}"),
    ]
)

retrieve_decision_chain = RETRIEVE_DECISION_PROMPT | retrieve_decider

In [ ]:
retrieve_decision_chain.invoke({"question": "What is 9 plus 10?"})

In [ ]:
retrieve_decision_chain.invoke({"question": "What reflection tokens does Self-RAG use and what does each one control?"})

### Discussion of the output

A trivial arithmetic question is graded `need_retrieval='no'`, while a question that clearly depends on the toy corpus's content is graded `need_retrieval='yes'`. This decision is made *before* the vector store is ever touched — the first place Self-RAG's self-reflection changes the control flow.

## Reflection workflow 2 — Per-document relevance grading

Similar in spirit to a CRAG-style grader, but self-contained and independently implemented here: each retrieved
chunk is graded individually as relevant or not to the current question.

In [ ]:
class RelevanceGrade(BaseModel):
    """Binary relevance grade for one retrieved document against a question."""

    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the document is relevant to answering the question, otherwise 'no'."
    )


relevance_grader = llm.with_structured_output(RelevanceGrade)

RELEVANCE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are grading whether a retrieved document is relevant to a user question. "
            "Grade 'yes' if it contains information that helps answer the question, even partially. "
            "Grade 'no' otherwise.",
        ),
        ("human", "Question:\n{question}\n\nRetrieved document:\n{document}"),
    ]
)

relevance_grading_chain = RELEVANCE_PROMPT | relevance_grader

In [ ]:
docs = retriever.invoke("What is Tree of Thoughts?")
for doc in docs:
    grade = relevance_grading_chain.invoke({"question": "What is Tree of Thoughts?", "document": doc.page_content})
    print(doc.metadata["topic"], "->", grade.binary_score)

## Reflection workflow 3 — Answer generation (with optional regeneration feedback)

The generation chain accepts an optional `feedback` field. On the first pass it is empty; if a later
self-reflection step (groundedness or utility) rejects the answer, the feedback explaining *why* is fed back in
so the regeneration attempt can actually correct the specific problem instead of blindly retrying.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

GENERATION_PROMPT = ChatPromptTemplate.from_template(
    """You are answering a question about agentic AI design patterns.
Use only the information in the provided context documents. If the context does not contain
enough information to answer confidently, say so plainly instead of guessing.

{feedback_block}

Question:
{question}

Context:
{context}

Answer:"""
)


def _format_docs(docs):
    if not docs:
        return "(no context documents were retrieved)"
    return "\n\n".join(f"[{d.metadata.get('topic', 'doc')}] {d.page_content}" for d in docs)


def _format_feedback(feedback):
    if not feedback:
        return ""
    return f"NOTE: a previous attempt at this answer was rejected for this reason — fix it this time:\n{feedback}"


generation_chain = (
    {
        "context": itemgetter("documents") | RunnableLambda(_format_docs),
        "question": itemgetter("question"),
        "feedback_block": itemgetter("feedback") | RunnableLambda(_format_feedback),
    }
    | GENERATION_PROMPT
    | llm
    | StrOutputParser()
)

In [ ]:
docs = retriever.invoke("What is Self-RAG and what do its reflection tokens control?")
answer = generation_chain.invoke({"question": "What is Self-RAG and what do its reflection tokens control?", "documents": docs, "feedback": ""})
print(answer)

## Reflection workflow 4 — Groundedness (support) check

This is the hallucination check: does the generated answer actually rest on facts present in the retrieved
documents, or did the model add unsupported claims?

In [ ]:
class GroundednessGrade(BaseModel):
    """Judgment on whether a generation is supported by its context documents."""

    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if every claim in the generation is supported by the context documents, 'no' otherwise."
    )
    feedback: str = Field(description="If 'no', a short explanation of which claims are unsupported.")


groundedness_grader = llm.with_structured_output(GroundednessGrade)

GROUNDEDNESS_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict fact-checker. Given a set of context documents and a generated answer, "
            "decide whether the answer is fully grounded in (supported by) the context. Flag any claim "
            "that is not backed by the context as unsupported.",
        ),
        ("human", "Context documents:\n{context}\n\nGenerated answer:\n{generation}"),
    ]
)

groundedness_chain = GROUNDEDNESS_PROMPT | groundedness_grader

In [ ]:
grade = groundedness_chain.invoke({"context": _format_docs(docs), "generation": answer})
grade

## Reflection workflow 5 — Utility check

Even a fully grounded answer can miss the point of the question (e.g. it discusses a tangentially related topic
instead of directly answering). This check grades whether the answer is actually useful for the question asked.

In [ ]:
class UtilityGrade(BaseModel):
    """Judgment on whether a generation actually addresses the user's question."""

    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the generation directly and usefully addresses the question, 'no' otherwise."
    )
    feedback: str = Field(description="If 'no', a short explanation of what the answer is missing.")


utility_grader = llm.with_structured_output(UtilityGrade)

UTILITY_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are grading whether a generated answer is useful — i.e. it directly and adequately "
            "addresses the user's original question, regardless of whether it is grounded in any context.",
        ),
        ("human", "Question:\n{question}\n\nGenerated answer:\n{generation}"),
    ]
)

utility_chain = UTILITY_PROMPT | utility_grader

In [ ]:
utility_chain.invoke({"question": "What is Self-RAG and what do its reflection tokens control?", "generation": answer})

## Define the Agent State Schema

`SelfRAGState` carries everything the graph's nodes read and write, plus an `iteration` counter (capped by
`max_iterations`) so that the groundedness-regenerate loop and the utility/relevance-driven re-retrieve loop are
both guaranteed to terminate.

In [ ]:
from typing import List, Optional
from typing_extensions import TypedDict


class SelfRAGState(TypedDict):
    """
    Represents the state of the Self-RAG agent during execution.

    Attributes:
        original_question: the user's original question, never overwritten.
        question: the (possibly rephrased) question currently driving retrieval/generation.
        need_retrieval: 'yes' / 'no' decision from the retrieve-decision reflection step.
        documents: the current list of retrieved (and relevance-filtered) context documents.
        generation: the current generated answer.
        groundedness_grade: 'yes' / 'no' — is the generation supported by the documents.
        groundedness_feedback: explanation to feed back into regeneration if not grounded.
        utility_grade: 'yes' / 'no' — does the generation usefully address the question.
        utility_feedback: explanation to feed back into re-retrieval if not useful.
        iteration: number of correction loops taken so far.
        max_iterations: hard cap on correction loops, to guarantee termination.
    """

    original_question: str
    question: str
    need_retrieval: str
    documents: List[Document]
    generation: str
    groundedness_grade: str
    groundedness_feedback: str
    utility_grade: str
    utility_feedback: str
    iteration: int
    max_iterations: int

## Plan the Agent Workflow Structure

Self-RAG's graph has four independent reflection-driven branch points (compare this to CRAG's one fallback
branch, or Adaptive RAG's one upfront route):

1. **`decide_retrieval`** → conditional edge to either `retrieve` or `generate` (skip retrieval entirely).
2. **`grade_relevance`** → conditional edge to either `generate` (some relevant docs found) or `rephrase_and_retry`
   (no relevant docs, and iterations remain) or `generate` anyway once the cap is hit.
3. **`grade_groundedness`** → conditional edge to either `grade_utility` (well supported) or back to `generate`
   for a stricter regeneration (not supported, iterations remain), or `grade_utility` once the cap is hit.
4. **`grade_utility`** → conditional edge to either `END` (useful) or `rephrase_and_retry` (not useful, iterations
   remain), or `END` once the cap is hit.

Next up we define the Python node functions for each stage, then wire the conditional edges.

## Create Node Functions

1. **decide_retrieval**: runs the retrieve-decision reflection step and records `need_retrieval`.
2. **retrieve**: retrieves the top-k candidate documents for the current (possibly rephrased) question.
3. **grade_relevance**: grades each retrieved document and keeps only the ones marked relevant.
4. **rephrase_and_retry**: rewrites the question and bumps the iteration counter — this single node is the
   shared re-entry point used both when relevance grading finds nothing useful and when the utility check fails.
5. **generate**: generates an answer from the current relevant documents (or from parametric knowledge only, if
   retrieval was skipped), optionally conditioned on groundedness feedback from a previous rejected attempt.
6. **grade_groundedness**: checks whether the generation is supported by the documents.
7. **grade_utility**: checks whether the generation actually answers the original question.

In [ ]:
def decide_retrieval(state: SelfRAGState):
    """Reflection step 1: decide whether retrieval is needed at all."""
    print("---REFLECT: DO WE NEED TO RETRIEVE?---")
    question = state["question"]
    decision = retrieve_decision_chain.invoke({"question": question})
    print(f"---DECISION: need_retrieval={decision.need_retrieval} ({decision.reasoning})---")
    return {"need_retrieval": decision.need_retrieval}

In [ ]:
def retrieve(state: SelfRAGState):
    """Retrieve candidate documents for the current question."""
    print("---RETRIEVE FROM VECTOR STORE---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents}

In [ ]:
def grade_relevance(state: SelfRAGState):
    """Reflection step 2: grade each retrieved document for relevance and keep only the relevant ones."""
    print("---REFLECT: GRADE DOCUMENT RELEVANCE---")
    question = state["question"]
    documents = state["documents"]

    relevant_docs = []
    for doc in documents:
        grade = relevance_grading_chain.invoke({"question": question, "document": doc.page_content})
        tag = "RELEVANT" if grade.binary_score == "yes" else "NOT RELEVANT"
        print(f"---GRADE: {doc.metadata.get('topic', 'doc')} -> {tag}---")
        if grade.binary_score == "yes":
            relevant_docs.append(doc)

    return {"documents": relevant_docs}

In [ ]:
def rephrase_and_retry(state: SelfRAGState):
    """Rewrite the question and record another correction-loop iteration.

    Shared re-entry point for two different failure modes: no relevant documents were found, or the
    generated answer failed the utility check. In both cases the fix is the same — try a differently
    phrased question and go around again.
    """
    print("---REWRITE QUESTION FOR ANOTHER RETRIEVAL PASS---")
    original_question = state["original_question"]
    current_question = state["question"]
    utility_feedback = state.get("utility_feedback", "")

    hint = f"\n\nPrevious attempt's utility feedback: {utility_feedback}" if utility_feedback else ""
    rewrite_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Rewrite the question to be more specific and better suited for retrieval from a small "
                "knowledge base about agentic AI design patterns. Keep the underlying intent the same.",
            ),
            ("human", "Original question:\n{question}{hint}"),
        ]
    )
    rewriter = rewrite_prompt | llm | StrOutputParser()
    new_question = rewriter.invoke({"question": current_question, "hint": hint})

    iteration = state.get("iteration", 0) + 1
    print(f"---ITERATION {iteration}: '{current_question}' -> '{new_question}'---")

    return {"question": new_question, "iteration": iteration}

In [ ]:
def generate(state: SelfRAGState):
    """Reflection step 3: generate an answer, optionally conditioned on prior groundedness feedback."""
    print("---GENERATE ANSWER---")
    question = state["original_question"]
    documents = state.get("documents", [])
    feedback = state.get("groundedness_feedback", "")

    answer = generation_chain.invoke({"question": question, "documents": documents, "feedback": feedback})
    return {"generation": answer}

In [ ]:
def grade_groundedness(state: SelfRAGState):
    """Reflection step 4: check whether the generation is supported by the retrieved documents.

    Skipped in spirit (auto-passed) when there are no documents at all — there is nothing to be
    grounded in, so this becomes purely a utility question instead.
    """
    print("---REFLECT: CHECK GROUNDEDNESS---")
    documents = state.get("documents", [])
    generation = state["generation"]

    if not documents:
        print("---NO DOCUMENTS TO GROUND IN - TREATING AS GROUNDED BY DEFAULT---")
        return {"groundedness_grade": "yes", "groundedness_feedback": ""}

    grade = groundedness_chain.invoke({"context": _format_docs(documents), "generation": generation})
    print(f"---GROUNDEDNESS: {grade.binary_score}---")
    return {"groundedness_grade": grade.binary_score, "groundedness_feedback": grade.feedback}

In [ ]:
def grade_utility(state: SelfRAGState):
    """Reflection step 5: check whether the generation actually addresses the original question."""
    print("---REFLECT: CHECK UTILITY---")
    question = state["original_question"]
    generation = state["generation"]

    grade = utility_chain.invoke({"question": question, "generation": generation})
    print(f"---UTILITY: {grade.binary_score}---")
    return {"utility_grade": grade.binary_score, "utility_feedback": grade.feedback}

## Build and Compile the Agent Graph

Four conditional-edge functions implement the four reflection-driven branch points described above. Each one
checks the relevant grade in state and, where applicable, the iteration cap, before deciding where to route
next.

In [ ]:
def route_after_retrieval_decision(state: SelfRAGState) -> str:
    if state["need_retrieval"] == "yes":
        return "retrieve"
    print("---DECISION: SKIPPING RETRIEVAL, GENERATING DIRECTLY---")
    return "generate"


def route_after_relevance_grading(state: SelfRAGState) -> str:
    has_relevant_docs = len(state.get("documents", [])) > 0
    iteration = state.get("iteration", 0)
    max_iterations = state.get("max_iterations", 2)

    if has_relevant_docs:
        return "generate"
    if iteration < max_iterations:
        print("---DECISION: NO RELEVANT DOCUMENTS, REPHRASE AND RETRY---")
        return "rephrase_and_retry"
    print("---DECISION: ITERATION CAP REACHED, GENERATING WITH WHATEVER WE HAVE---")
    return "generate"


def route_after_groundedness(state: SelfRAGState) -> str:
    iteration = state.get("iteration", 0)
    max_iterations = state.get("max_iterations", 2)

    if state["groundedness_grade"] == "yes":
        return "grade_utility"
    if iteration < max_iterations:
        print("---DECISION: NOT GROUNDED, REGENERATE---")
        # Regenerating counts as a correction-loop iteration too, to keep the cap meaningful.
        state["iteration"] = iteration + 1
        return "generate"
    print("---DECISION: ITERATION CAP REACHED, ACCEPTING UNGROUNDED ANSWER AS FINAL---")
    return "grade_utility"


def route_after_utility(state: SelfRAGState) -> str:
    iteration = state.get("iteration", 0)
    max_iterations = state.get("max_iterations", 2)

    if state["utility_grade"] == "yes":
        return "end"
    if iteration < max_iterations:
        print("---DECISION: NOT USEFUL, REPHRASE AND RETRY---")
        return "rephrase_and_retry"
    print("---DECISION: ITERATION CAP REACHED, ACCEPTING BEST-EFFORT ANSWER---")
    return "end"

In [ ]:
from langgraph.graph import END, StateGraph

self_rag = StateGraph(SelfRAGState)

self_rag.add_node("decide_retrieval", decide_retrieval)
self_rag.add_node("retrieve", retrieve)
self_rag.add_node("grade_relevance", grade_relevance)
self_rag.add_node("rephrase_and_retry", rephrase_and_retry)
self_rag.add_node("generate", generate)
self_rag.add_node("grade_groundedness", grade_groundedness)
self_rag.add_node("grade_utility", grade_utility)

self_rag.set_entry_point("decide_retrieval")

# Reflection point 1: retrieve or skip straight to generation
self_rag.add_conditional_edges(
    "decide_retrieval",
    route_after_retrieval_decision,
    {"retrieve": "retrieve", "generate": "generate"},
)

self_rag.add_edge("retrieve", "grade_relevance")

# Reflection point 2: enough relevant documents, or rephrase and retry retrieval
self_rag.add_conditional_edges(
    "grade_relevance",
    route_after_relevance_grading,
    {"generate": "generate", "rephrase_and_retry": "rephrase_and_retry"},
)

self_rag.add_edge("rephrase_and_retry", "retrieve")

self_rag.add_edge("generate", "grade_groundedness")

# Reflection point 3: grounded enough to move on, or regenerate
self_rag.add_conditional_edges(
    "grade_groundedness",
    route_after_groundedness,
    {"grade_utility": "grade_utility", "generate": "generate"},
)

# Reflection point 4: useful enough to finish, or rephrase and retry from retrieval
self_rag.add_conditional_edges(
    "grade_utility",
    route_after_utility,
    {"end": END, "rephrase_and_retry": "rephrase_and_retry"},
)

self_rag = self_rag.compile()

In [ ]:
from IPython.display import Image, display, Markdown

display(Image(self_rag.get_graph().draw_mermaid_png()))

## Test the Self-RAG System

### Case 1 — a question that doesn't need retrieval at all

The retrieve-decision step should short-circuit straight to `generate`, skipping the vector store entirely.

In [ ]:
result = self_rag.invoke({
    "original_question": "What is 9 plus 10?",
    "question": "What is 9 plus 10?",
    "iteration": 0,
    "max_iterations": 2,
})
display(Markdown(result["generation"]))

### Case 2 — a question well covered by the toy corpus

Expect: retrieve, relevant docs found, generation grounded and useful on the first pass.

In [ ]:
result = self_rag.invoke({
    "original_question": "What is Self-RAG and what do its reflection tokens control?",
    "question": "What is Self-RAG and what do its reflection tokens control?",
    "iteration": 0,
    "max_iterations": 2,
})
display(Markdown(result["generation"]))

### Case 3 — a question with nothing relevant in the toy corpus

Expect: retrieval finds nothing relevant, the relevance-grading reflection triggers a rephrase-and-retry loop, and once the iteration cap is hit the system generates a best-effort (and, ideally, honest) answer rather than looping forever.

In [ ]:
result = self_rag.invoke({
    "original_question": "What is the weather forecast for Paris tomorrow?",
    "question": "What is the weather forecast for Paris tomorrow?",
    "iteration": 0,
    "max_iterations": 2,
})
display(Markdown(result["generation"]))

### Discussion of the output

Case 1 demonstrates the retrieve-or-not reflection point acting on its own, before any document is ever touched — something neither CRAG nor Adaptive RAG do (both always retrieve from some source). Case 2 shows the full happy path through all four reflection points passing on the first try. Case 3 shows the bounded self-correction loop: the relevance grader correctly finds nothing usable in a knowledge base about agentic AI patterns for a weather question, the rephrase-and-retry node fires, and the iteration cap guarantees the graph still terminates with an honest best-effort answer instead of looping indefinitely.

## Summary — Key Takeaways

- **Self-RAG's defining idea is self-reflection at multiple points, not a single router or a single fallback.**
  This notebook implements four independent structured-output judgments from the same LLM — retrieve-decision,
  per-document relevance, groundedness, and utility — each wired to its own conditional edge in the LangGraph
  `StateGraph`.
- **Retrieval is conditional, not automatic.** The very first node can decide external context isn't needed at
  all and route straight to generation — a capability CRAG and Adaptive RAG in this folder don't have, since
  both of those always retrieve from some source.
- **Correction is self-directed, not tool-directed.** Where CRAG and Adaptive RAG recover from bad retrieval by
  falling back to a *different* external tool (a web search), Self-RAG recovers by having the model rephrase its
  own query and re-run its own retrieval/generation — the loop is entirely internal to the model's judgments.
- **Groundedness and utility are graded separately**, because they can fail independently: an answer can be
  perfectly grounded in the retrieved documents yet fail to actually address the question, or it can directly
  address the question while making claims the documents don't support. Splitting them into two reflection
  points lets each failure mode trigger its own, more targeted correction (regenerate vs. re-retrieve).
- **An iteration cap (`max_iterations` in the state) is what makes the self-correction loops safe.** Both the
  relevance-driven and utility-driven rephrase-and-retry loops, and the groundedness-driven regenerate loop,
  check the same counter before looping again, guaranteeing the graph always terminates with a best-effort
  answer rather than looping forever on a query the toy corpus simply can't answer.
- **The whole system runs on this repo's `helpers.get_llm` / `helpers.get_embeddings` factory**, so it stays
  platform-aware and never instantiates a provider client directly, and it is fully self-contained — a small
  in-notebook toy corpus embedded into an in-memory Chroma store — so it can run end to end without any external
  downloads or a web search API key.